In [ ]:
import os
import json
import sys
import re
import threading
from concurrent.futures import ThreadPoolExecutor
# Add the project root to the path to allow importing utils
sys.path.append('../../')
import utils.utils as utils
from importlib import reload
reload(utils)

<module 'utils.utils' from '/Users/jlee0/Desktop/research/fine-tuning-or-retrieval/notebooks/FT/../../utils/utils.py'>

In [ ]:
# --- 1. Setup ---
PAPER_NAME = "DPO"
PAPER_FILE_PATH = f'../../data/arxiv/cleaned_{PAPER_NAME}.txt'
OUTPUT_DIR = f"../../data/arxiv/prior_knowledge/{PAPER_NAME}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Reading paper from: {PAPER_FILE_PATH}")
with open(PAPER_FILE_PATH, 'r') as f:
    paper_content = f.read()

# --- 2. Generate list of chapters ---
print("Generating list of prerequisite chapters...")
prompt_chapters = {
    'system': """### Instructions
You are an expert curriculum designer. Based on the provided research paper, create a list of textbook chapters that would provide all the necessary prior knowledge to understand this paper. The chapters should not contain the novel ideas presented in the paper itself, but rather the foundational concepts upon which the paper is built.

For each chapter, provide:
- A `title`.
- A general `description` of what the chapter covers.
- A list of `subtopics` that should be included.

### Output Format
Provide the output as a JSON object with a single key "chapters", which is a list of chapter dictionaries.
Example:
{
  "chapters": [
    {
      "title": "Chapter 1: Introduction to Probability Theory",
      "description": "This chapter covers the basics of probability...",
      "subtopics": ["Random Variables", "Probability Distributions", "Bayes' Theorem"]
    }
  ]
}""",
    'user': f"### Research Paper\n{paper_content}"
}

# Although we ask for JSON, the model might return it as a string.
# We set return_json=False and parse it manually for robustness.
response_chapters_str = utils.query_llm(
    prompt_chapters, 
    model='gpt-5-mini', 
    system_prompt_included=True, 
    return_json=True, 
    max_tokens=5000
)

Reading paper from: ../../data/arxiv/cleaned_DPO.txt
Generating list of prerequisite chapters...


In [5]:
# --- 3. Parse chapter list ---
chapters_list = []
try:
    response_json = json.loads(response_chapters_str)
    chapters_list = response_json.get('chapters', [])
    print(f"Successfully parsed {len(chapters_list)} chapters.")
    for i, chapter in enumerate(chapters_list):
        print(f"  {i+1}. {chapter.get('title', 'Untitled Chapter')}")
        print(f"     Description: {chapter.get('description', 'No description')}")
        print(f"     Subtopics: {chapter.get('subtopics', [])}")
except (json.JSONDecodeError, AttributeError) as e:
    print(f"Failed to parse chapter list from LLM response: {e}")
    print("Response was:\n", response_chapters_str)

Successfully parsed 17 chapters.
  1. Chapter 1: Probability and Information Theory for Machine Learning
     Description: Covers the probabilistic and information-theoretic tools needed to reason about language models, rewards, and divergences used in alignment and RLHF.
     Subtopics: ['Random variables, expectations, and log-likelihoods', 'Cross-entropy and binary cross-entropy losses', 'Logistic (sigmoid) and logit functions', 'KL divergence (forward vs. reverse) and properties', 'Entropy and mutual information (intuition)', 'Log-sum-exp trick and numerical stability', 'Partition functions and normalization constants', 'Jensen’s inequality and convexity basics']
  2. Chapter 2: Optimization and Loss Functions in Machine Learning
     Description: Introduces core optimization techniques and loss formulations relevant to training language models and reward models from preference data.
     Subtopics: ['Gradient descent and stochastic gradient descent (SGD)', 'Learning rate schedules

In [18]:
# --- 4. Generate content for each chapter ---
if chapters_list:
    
    def generate_chapter(chapter_info, chapter_index):
        """Generate content for a single chapter"""
        chapter_title = chapter_info.get('title', f"Chapter {chapter_index+1}")
        chapter_description = chapter_info.get('description', '')
        chapter_subtopics = chapter_info.get('subtopics', [])
        
        print(f"Generating content for: {chapter_title}")

        subtopics_str = "\n".join([f"- {s}" for s in chapter_subtopics])

        prompt_content = {
            'system': """### Instructions
You will be given a chapter title, description, and subtopics and, based on those topics, your job is to write a detailed, cohesive textbook chapter addressed to a college student who is learning this material for the first time. 

The chapter should be comprehensive and suitable for someone learning this material to understand research papers in the field. Begin with an introduction to the chapter, then cover each subtopic in turn. Don't just briefly describe the subtopics, but rather elaborate on the concepts at full length and explain them with a focus on intuition. Spell everything out clearly so there is no ambiguity. Dedicate multiple paragraphs to each subtopic. Write in full prose, rather than bullet points. 

Separate each subtopic with a section header "#".

Also, please write all mathematical notation in LaTeX only e.g. "$x^2$" or "$\pi$". Do not use unicode mathematical characters e.g. "π".""",
            'user': f"""### Chapter Title
{chapter_title}

### Chapter Description
{chapter_description}

### Subtopics to Cover
{subtopics_str}"""
        }

        chapter_content = utils.query_llm(
            prompt_content, 
            model='gpt-5-mini', 
            system_prompt_included=True, 
            reasoning_effort = "low",
            max_tokens=10000
        )

        # Truncate the last sentence if it's incomplete
        if chapter_content and not chapter_content.strip().endswith(('.', '!', '?', '"', '`')):
            last_period_index = chapter_content.rfind('.')
            if last_period_index != -1:
                chapter_content = chapter_content[:last_period_index+1]
        
        # --- 5. Tokenize and split the chapter ---
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-1124-7B")
        
        # Split chapter into sections (by # headers)
        import re
        sections = re.split(r'\n(?=#)', chapter_content)
        
        # Group sections into chunks of at most 3072 tokens
        chunks = []
        current_chunk = []
        current_token_count = 0
        
        for section in sections:
            section_tokens = len(tokenizer.encode(section))
            
            # If adding this section would exceed the limit, save current chunk
            if current_chunk and current_token_count + section_tokens > 3072:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = [section]
                current_token_count = section_tokens
            else:
                current_chunk.append(section)
                current_token_count += section_tokens
        
        # Add the last chunk if it exists
        if current_chunk:
            chunks.append('\n\n'.join(current_chunk))
        
        # Save chunks to files
        saved_paths = []
        for chunk_idx, chunk in enumerate(chunks):
            if len(chunks) == 1:
                # Single chunk, use original naming
                output_path = os.path.join(OUTPUT_DIR, f"chapter_{chapter_index+1}.txt")
            else:
                # Multiple chunks, use suffix naming
                suffix = chr(ord('a') + chunk_idx)  # a, b, c, etc.
                output_path = os.path.join(OUTPUT_DIR, f"chapter_{chapter_index+1}_{suffix}.txt")
            
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(chunk)
            
            saved_paths.append(output_path)
            token_count = len(tokenizer.encode(chunk))
            print(f"Saved: {output_path} ({token_count} tokens)")
        
        return saved_paths

    # Generate all chapters in parallel
    with ThreadPoolExecutor(max_workers=min(len(chapters_list), 5)) as executor:
        futures = [executor.submit(generate_chapter, chapter_info, i) 
                  for i, chapter_info in enumerate(chapters_list)]
        
        # Wait for all chapters to complete
        all_saved_paths = []
        for future in futures:
            try:
                result = future.result()
                all_saved_paths.extend(result)
            except Exception as e:
                print(f"Error generating chapter: {e}")

    print(f"\nAll chapters generated and saved successfully. Total files: {len(all_saved_paths)}")
else:
    print("No chapters were generated. Exiting.")


Generating content for: Chapter 1: Probability and Information Theory for Machine Learning
Generating content for: Chapter 2: Optimization and Loss Functions in Machine Learning
Generating content for: Chapter 3: Autoregressive Language Modeling with Transformers
Generating content for: Chapter 4: Transformer Architecture Essentials
Generating content for: Chapter 5: Supervised Fine-Tuning and Instruction Tuning
Saved: ../../data/arxiv/prior_knowledge/DPO/chapter_5_a.txt (2975 tokens)
Saved: ../../data/arxiv/prior_knowledge/DPO/chapter_5_b.txt (1005 tokens)
Generating content for: Chapter 6: Human Preference Data and Annotation
Saved: ../../data/arxiv/prior_knowledge/DPO/chapter_4_a.txt (2923 tokens)
Saved: ../../data/arxiv/prior_knowledge/DPO/chapter_4_b.txt (1574 tokens)
Generating content for: Chapter 7: Preference Models: Bradley–Terry and Plackett–Luce
Saved: ../../data/arxiv/prior_knowledge/DPO/chapter_3_a.txt (2619 tokens)
Saved: ../../data/arxiv/prior_knowledge/DPO/chapter_3_b.